# Experiment 3.0.2 — Hidden multi-τ architecture comparison

This notebook is **analysis-only**. Formal execution is:

`Slurm training array → Slurm evaluation array → finalizer → this notebook`

Architectures:

- **A**: L1=(3), L2=(3), L3=(3); reused from Experiment 3.0.1 `paired_split_v2`.
- **B**: L1=(2,3), L2=(2,3), L3=(3).
- **C**: L1=(2,3,4), L2=(2,3,4), L3=(3).
- **D**: L1=(2,3), L2=(2,3,4,5), L3=(3).
- **E**: L1=(2,3), L2=(2,3,4), L3=(3).

All architectures use the same 30→128→128→64 SNN backbone, the same fixed user split, three objectives (`timestep_ce`, `relative10_sequence_ce`, `fixed250_sequence_ce`) and seeds 11/23/101. L3 is intentionally fixed at shift 3 so the experiment isolates hidden-layer temporal organization.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for p in (start, *start.parents):
        if (p / 'snn').is_dir() and (p / 'notebooks').is_dir():
            return p
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_3_0_2_hidden_multitau_architecture_comparison' / 'hidden_multitau_v1'
print('Repository root:', REPO_ROOT)
print('Results dir:', RESULTS_DIR)


In [ ]:
FILES = {
    'architectures': 'experiment_3_0_2_architectures.csv',
    'native': 'experiment_3_0_2_native_results.csv',
    'history': 'experiment_3_0_2_history.csv',
    'layer_probes': 'experiment_3_0_2_layer_probes.csv',
    'subgroup_probes': 'experiment_3_0_2_tau_subgroup_probes.csv',
    'firing': 'experiment_3_0_2_firing_rates.csv',
    'native_summary': 'experiment_3_0_2_native_summary.csv',
    'layer_probe_summary': 'experiment_3_0_2_layer_probe_summary.csv',
    'subgroup_probe_summary': 'experiment_3_0_2_tau_subgroup_probe_summary.csv',
    'firing_summary': 'experiment_3_0_2_firing_rate_summary.csv',
}
missing = [name for name in FILES.values() if not (RESULTS_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Run the Slurm pipeline/finalizer first. Missing: ' + ', '.join(missing))

tables = {key: pd.read_csv(RESULTS_DIR / name) for key, name in FILES.items()}
native = tables['native']
history = tables['history']
layer_probes = tables['layer_probes']
subgroup_probes = tables['subgroup_probes']
firing = tables['firing']
native_summary = tables['native_summary']
layer_probe_summary = tables['layer_probe_summary']
subgroup_probe_summary = tables['subgroup_probe_summary']
firing_summary = tables['firing_summary']

assert len(native) == 45, len(native)
display(tables['architectures'])
display(native_summary.sort_values(['objective', 'mean_test_balanced_accuracy'], ascending=[True, False]))


## 1. Native test balanced accuracy
Compare A–E under each native training objective.

In [ ]:
objectives = ['timestep_ce', 'relative10_sequence_ce', 'fixed250_sequence_ce']
architectures = ['A', 'B', 'C', 'D', 'E']
x = np.arange(len(architectures))
width = 0.24
fig, ax = plt.subplots(figsize=(11, 5))
for i, objective in enumerate(objectives):
    d = native_summary[native_summary.objective == objective].set_index('architecture').reindex(architectures)
    ax.bar(x + (i - 1) * width, d.mean_test_balanced_accuracy, width, yerr=d.sd_test_balanced_accuracy, capsize=3, label=objective)
ax.set_xticks(x, architectures)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Experiment 3.0.2 — native test BA')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


## 2. Layer-wise frozen linear probes
Each trained SNN is frozen. L1/L2/L3 are independently evaluated with full-count and fixed-250-ms representations followed by StandardScaler + LogisticRegression.

In [ ]:
for objective in objectives:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for ax, probe_type in zip(axes, ['full_count', 'fixed250']):
        d = layer_probe_summary[(layer_probe_summary.objective == objective) & (layer_probe_summary.probe_type == probe_type)]
        for architecture in architectures:
            q = d[d.architecture == architecture].set_index('layer').reindex(['L1', 'L2', 'L3'])
            ax.errorbar(['L1', 'L2', 'L3'], q.mean_probe_test_balanced_accuracy, yerr=q.sd_probe_test_balanced_accuracy, marker='o', capsize=3, label=architecture)
        ax.set_title(f'{objective} — {probe_type}')
        ax.set_ylabel('Frozen probe test BA')
        ax.grid(alpha=0.25)
    axes[-1].legend(title='Architecture')
    plt.tight_layout()
    plt.show()


## 3. L3 fixed250 probe versus native classifier
This separates native readout quality from the discriminative information contained in the final 64-neuron feature layer.

In [ ]:
l3 = layer_probe_summary[(layer_probe_summary.layer == 'L3') & (layer_probe_summary.probe_type == 'fixed250')]
comparison = native_summary.merge(
    l3[['objective', 'architecture', 'mean_probe_test_balanced_accuracy', 'sd_probe_test_balanced_accuracy']],
    on=['objective', 'architecture'],
    how='left',
)
display(comparison[['objective', 'architecture', 'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy', 'mean_probe_test_balanced_accuracy', 'sd_probe_test_balanced_accuracy']].sort_values(['objective', 'mean_probe_test_balanced_accuracy'], ascending=[True, False]))


## 4. Whole-layer firing rates
Test-set firing rates are normalized as spikes / valid timestep / neuron.

In [ ]:
d = firing_summary[(firing_summary['split'] == 'test') & (firing_summary.scope == 'whole_layer')]
for objective in objectives:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    q = d[d.objective == objective]
    for architecture in architectures:
        z = q[q.architecture == architecture].set_index('layer').reindex(['L1', 'L2', 'L3'])
        ax.plot(['L1', 'L2', 'L3'], z.mean_firing_rate, marker='o', label=architecture)
    ax.set_title(f'{objective} — test firing rate')
    ax.set_ylabel('Spikes / valid timestep / neuron')
    ax.grid(alpha=0.25)
    ax.legend(title='Architecture')
    plt.show()


## 5. τ-subgroup probes and firing rates
Only genuinely multi-τ layers are included. Compare subgroups mainly **within the same layer/configuration**, because subgroup feature dimensions can differ across architectures.

In [ ]:
if len(subgroup_probe_summary):
    display(subgroup_probe_summary.sort_values(['objective', 'architecture', 'layer', 'probe_type', 'shift']))

    for objective in objectives:
        d = subgroup_probe_summary[(subgroup_probe_summary.objective == objective) & (subgroup_probe_summary.probe_type == 'fixed250')]
        fig, ax = plt.subplots(figsize=(11, 5))
        for (architecture, layer), q in d.groupby(['architecture', 'layer']):
            q = q.sort_values('shift')
            ax.plot(q['shift'], q.mean_probe_test_balanced_accuracy, marker='o', label=f'{architecture}-{layer}')
        ax.set_xticks([2, 3, 4, 5])
        ax.set_xlabel('shift_syn subgroup')
        ax.set_ylabel('Subgroup fixed250 probe test BA')
        ax.set_title(f'{objective} — τ-subgroup representation')
        ax.grid(alpha=0.25)
        ax.legend(ncol=2)
        plt.show()


In [ ]:
sub_fr = firing_summary[(firing_summary['split'] == 'test') & (firing_summary.scope == 'tau_subgroup')]
display(sub_fr.sort_values(['objective', 'architecture', 'layer', 'shift']))


## 6. Validation trajectories
A uses the original Experiment 3.0.1 history; B–E use the new Experiment 3.0.2 histories.

In [ ]:
for objective in objectives:
    fig, ax = plt.subplots(figsize=(10, 5))
    q = history[history.objective == objective]
    for architecture in architectures:
        a = q[q.architecture == architecture]
        curve = a.groupby('epoch').val_balanced_accuracy.agg(['mean', 'std']).reset_index()
        ax.plot(curve.epoch, curve['mean'], label=architecture)
        ax.fill_between(curve.epoch, curve['mean'] - curve['std'].fillna(0), curve['mean'] + curve['std'].fillna(0), alpha=0.12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation balanced accuracy')
    ax.set_title(f'{objective} — validation BA vs epoch')
    ax.grid(alpha=0.25)
    ax.legend(title='Architecture')
    plt.show()


## 7. Rankings


In [ ]:
print('Native ranking')
display(native_summary.sort_values('mean_test_balanced_accuracy', ascending=False)[['objective', 'architecture', 'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy', 'mean_train_test_ba_gap']])

print('L3 fixed250 frozen-probe ranking')
display(l3.sort_values('mean_probe_test_balanced_accuracy', ascending=False)[['objective', 'architecture', 'mean_probe_test_balanced_accuracy', 'sd_probe_test_balanced_accuracy']])
